# Replaying experiments

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

A **replay** is a simulation whose agents are driven by recorded data instead of by a model. The
arena, the rendering, the camera and the video export are the ones you would use for a simulation -
only the movement comes from a file.

That makes it two useful things at once. It is the most direct way to *look* at a dataset : whether
the tracking is sound, whether an animal was lost, whether the arena dimensions are right. And it
is the way to put a recording and a simulation side by side in the same visual language, which is
what the [GA worked example](../5_optimization_and_evaluation/ga_turner_noise_optimization.ipynb)
does.

**What you will be able to do afterwards**

- Replay any imported dataset, whole or sliced in time.
- Transpose all trajectories to a common origin to compare dispersal rather than position.
- Follow one animal, fixing a midline point or a body segment to the centre of the screen.
- Reconstruct the body from a chosen number of segments.
- Collapse a whole track into one overlapped image.
- Write any of the above to a video file.

**Prerequisites** : [Importing experimental data](import_datasets.ipynb), or any dataset already
registered under a reference ID. The default reference dataset ships with the package.

**Cost** : nothing while the switches are off. Each replay is a real simulation, and rendering to
video takes minutes.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_REPLAY_DEMOS` | `False` | the replays themselves |
| `SAVE_MEDIA` | `False` | writing them to video files instead of only displaying |

## Setup

In [1]:
%matplotlib inline

%load_ext param.ipython

import larvaworld
from larvaworld.lib import reg
from larvaworld.lib.reg.generators import ReplayConf, ReplayConfGroup, ReplayConfUnit
from larvaworld.lib.sim import ReplayRun

larvaworld.VERBOSE = 1

# Tutorial safety switches
RUN_REPLAY_DEMOS = False  # runs the replay simulations
SAVE_MEDIA = False  # writes videos
MEDIA_DIR = "./media/replay"

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : How a replay is configured

Three classes, split by the scope of what they control :

- **`ReplayConfGroup`** - what applies to the whole population : which agents to show, whether to
  transpose the coordinates, which midline point counts as *the* position, which environment to
  draw them in.
- **`ReplayConfUnit`** - what only makes sense for a single animal : the close-up view, and fixing a
  midline point or a body segment to the centre of the screen.
- **`ReplayConf`** - both of the above, plus which dataset to replay, the time range, the overlap
  image, and how many segments to rebuild the body from.

You always instantiate `ReplayConf`; the other two exist because the split is what the Portal's
replay app and the command line use to group their options.

In [2]:
%params ReplayConfGroup

In [3]:
%params ReplayConfUnit

In [4]:
%params ReplayConf

## Section 2 : Choosing a dataset

A replay needs a dataset, given either by its reference ID or by the directory it lives in. Any
dataset you imported with a `refID` is in this list.

In [5]:
refIDs = reg.conf.Ref.confIDs
print(f"{len(refIDs)} reference datasets :")
print(refIDs)

11 reference datasets :
['Chris.larvae_single', 'DeepLabCut.TopDown-2023-07-05', 'DeepLabCut.TopDown-2024-02-17', 'FeedingState.Fed', 'FeedingState.Starved', 'FeedingState.Sucrose', 'FreeExploration.pooled', 'FreeExploration.single_dish', 'exploration.30controls', 'exploration.dish01', 'exploration.dish02']


In [6]:
refID = reg.default_refID
d = reg.loadRef(id=refID, load=RUN_REPLAY_DEMOS)

print(f"Replaying {refID!r}")
print(
    f"  {d.config.N} larvae, dt={d.config.dt} s, duration={d.config.duration:.2f} min"
)
print(
    f"  arena {d.config.env_params.arena.dims} m, {d.config.env_params.arena.geometry}"
)
print(
    f"  midline points : {d.config.Npoints}, tracked point index : {d.config.point_idx}"
)

Loaded existing conf larvae_single
Loaded stored reference dataset : Chris.larvae_single
Loaded existing conf TopDown-2023-07-05
Loaded stored reference dataset : DeepLabCut.TopDown-2023-07-05
Loaded existing conf TopDown-2024-02-17
Loaded stored reference dataset : DeepLabCut.TopDown-2024-02-17
Loaded existing conf Fed
Loaded stored reference dataset : FeedingState.Fed
Loaded existing conf Starved
Loaded stored reference dataset : FeedingState.Starved
Loaded existing conf Sucrose
Loaded stored reference dataset : FeedingState.Sucrose
Loaded existing conf pooled
Loaded stored reference dataset : FreeExploration.pooled
Loaded existing conf single_dish
Loaded stored reference dataset : FreeExploration.single_dish
Loaded existing conf 30controls
Loaded stored reference dataset : exploration.30controls


Loaded existing conf dish01
Loaded stored reference dataset : exploration.dish01
Loaded existing conf dish02
Loaded stored reference dataset : exploration.dish02
Loaded existing conf 30controls
Loaded stored reference dataset : exploration.30controls
Replaying 'exploration.30controls'
  30 larvae, dt=0.0625 s, duration=3.00 min
  arena (0.15, 0.15) m, circular
  midline points : 12, tracked point index : 9


Note the `load` argument : the configuration of a dataset is cheap to read, the timeseries is not.
`reg.loadRef(id, load=False)` gives you the former without touching the HDF5 file, which is why the
cell above only loads the data when the replays are actually going to run.

## Section 3 : The replay modes

Each entry below is one way of looking at the same recording. They are collected in a dictionary so
that the running code stays identical and only the configuration changes - which is also how you
would script a batch of them.

| mode | what it shows | why you would use it |
|---|---|---|
| `normal` | the first minute, as recorded | a first look, and a check that the import is sane |
| `dispersal` | every track moved to start at the origin | compares the *shape and extent* of paths, not their position |
| `fixed_point` | one larva, midline point 6 pinned to the centre | isolates body bending from translation |
| `fixed_segment` | the same, with the rear segment also aligned | isolates bending of one half of the body |
| `fixed_overlap` | the whole track collapsed into one image | shows the range of postures at a glance |
| `2segs` | the body redrawn from 2 segments | the coarse body the model uses - directly comparable to a simulation |
| `all_segs` | the body redrawn from all 11 segments | uses every tracked midline point |

In [7]:
replay_confs = {
    "normal": {"time_range": (0, 60)},
    "dispersal": {"transposition": "origin"},
    "fixed_point": {
        "agent_ids": [0],
        "close_view": True,
        "fix_point": 6,
        "time_range": (80, 100),
    },
    "fixed_segment": {
        "agent_ids": [0],
        "close_view": True,
        "fix_point": 6,
        "fix_segment": "rear",
        "time_range": (100, 130),
    },
    "fixed_overlap": {
        "agent_ids": [0],
        "close_view": True,
        "fix_point": 6,
        "fix_segment": "front",
        "overlap_mode": True,
    },
    "2segs": {"draw_Nsegs": 2, "time_range": (80, 100)},
    "all_segs": {"draw_Nsegs": 11, "time_range": (80, 100)},
}

for mode, kws in replay_confs.items():
    print(f"{mode:16s} {kws}")

normal           {'time_range': (0, 60)}
dispersal        {'transposition': 'origin'}
fixed_point      {'agent_ids': [0], 'close_view': True, 'fix_point': 6, 'time_range': (80, 100)}
fixed_segment    {'agent_ids': [0], 'close_view': True, 'fix_point': 6, 'fix_segment': 'rear', 'time_range': (100, 130)}
fixed_overlap    {'agent_ids': [0], 'close_view': True, 'fix_point': 6, 'fix_segment': 'front', 'overlap_mode': True}
2segs            {'draw_Nsegs': 2, 'time_range': (80, 100)}
all_segs         {'draw_Nsegs': 11, 'time_range': (80, 100)}


## Section 4 : Running them

`ReplayRun` takes the configuration, an ID and the display settings, and `run()` does the work. One
helper is enough for every mode.

In [8]:
def run_replay(mode):
    """Replay the reference dataset in one of the modes defined above."""
    p = ReplayConf(refID=refID, **replay_confs[mode]).nestedConf

    screen_kws = {
        "show_display": False,
        "vis_mode": "video" if SAVE_MEDIA else None,
        "save_video": SAVE_MEDIA,
        "media_dir": f"{MEDIA_DIR}/{mode}",
        "video_file": f"{refID}_replay_{mode}",
    }

    rep = ReplayRun(
        parameters=p,
        id=f"{refID}_replay_{mode}",
        dir=f"{MEDIA_DIR}/{mode}",
        screen_kws=screen_kws,
    )
    return rep.run()

### The population

Straight replay first, then the same tracks transposed to a common origin. The second view removes
the arbitrary starting position of each animal, so what remains is how far and how directly each
one travelled - which is what dispersal means.

In [9]:
if RUN_REPLAY_DEMOS:
    run_replay("normal")
else:
    print("Set RUN_REPLAY_DEMOS = True to run the replays.")

Set RUN_REPLAY_DEMOS = True to run the replays.


In [10]:
if RUN_REPLAY_DEMOS:
    run_replay("dispersal")

### Reconstructed bodies

The tracked midline has many points; a model larva has few segments. Redrawing the recording with
2 or 11 segments shows what is lost at each level of abstraction - and the 2-segment version is
exactly the body a `explorer`-style model has, which is what makes real and simulated animals
visually comparable.

In [11]:
if RUN_REPLAY_DEMOS:
    run_replay("2segs")

In [12]:
if RUN_REPLAY_DEMOS:
    run_replay("all_segs")

### One animal, up close

Fixing a midline point to the centre of the screen removes translation entirely, leaving only what
the body is doing. Adding `fix_segment` aligns one half of the body as well, so that bending is
measured against a fixed axis.

In [13]:
if RUN_REPLAY_DEMOS:
    run_replay("fixed_point")

In [14]:
if RUN_REPLAY_DEMOS:
    run_replay("fixed_segment")

And `overlap_mode` collapses the whole sequence into a single image : every posture the animal
adopted, drawn on top of each other. It is the quickest way to see the range of bending a larva
actually used.

In [15]:
if RUN_REPLAY_DEMOS:
    run_replay("fixed_overlap")

## Where to go next

- [Importing experimental data](import_datasets.ipynb) - getting your own recordings into a state
  where they can be replayed.
- [Worked example: turner noise](../5_optimization_and_evaluation/ga_turner_noise_optimization.ipynb) -
  replays of real and simulated animals side by side.
- Reference : [Dataset replay](../../working_with_larvaworld/replay.md) and
  [Keyboard controls](../../visualization/keyboard_controls.md) for what you can do while a replay
  is on screen.